# End-to-end forecasting: daily demand with a rolling-origin backtest

Third of three notebooks. Forecasting is not "regression with a date column", and the two
mistakes that matter most are structural:

* **the split.** A random split trains on next week to predict last week. SKSurrogate
  refuses it, and this notebook shows the refusal;
* **the baseline.** A forecast that cannot beat seasonal naive has not earned the word
  "forecast". MASE makes that comparison explicit: 1.0 *is* seasonal naive, by definition.

The series is two and a half years of daily demand with weekly and annual seasonality, a
promotion flag that is only known at the origin, and — deliberately — gaps in both columns.

## What you will see

| Stage | SKSurrogate pieces |
| --- | --- |
| Declaring the temporal contract | `TemporalSpec`, `validate_temporal_frame` |
| Refusing a leakage-prone split | `mltrack.SetCV` guarded by `TEMPORAL_CV_SPLITTERS` |
| Leakage-safe features | `ForecastFeatureBuilder`, `make_supervised`, `assert_no_leakage` |
| Walk-forward evaluation against baselines | `mltrack.backtest`, `BASELINE_METHODS` |
| Interval calibration from backtest residuals | `backtest(alpha=..., interval_method="asymmetric")`, `interval_diagnostics` |
| Monitoring | `drift_report` in window mode |
| Deployment and roll-forward serving | `ModelBundle(temporal=...)`, `serve_forecast`, `predict_forecast` |

## How to run

```bash
jupyter lab examples/notebooks/03_time_series_end_to_end.ipynb
```

Offline by default; `SKSURROGATE_NOTEBOOK_DATA=openml` switches to a real daily series.
Artifacts land in `forecasting-report/`.

In [ ]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import Pipeline

REPO_ROOT = Path.cwd()
NOTEBOOK_DIR = REPO_ROOT / "examples" / "notebooks"
if not (NOTEBOOK_DIR / "notebook_data.py").exists():
    NOTEBOOK_DIR = REPO_ROOT  # launched from inside examples/notebooks
sys.path.insert(0, str(NOTEBOOK_DIR.resolve()))

import sklearn

import SKSurrogate
from SKSurrogate import (
    BASELINE_METHODS,
    DeploymentApprovalGate,
    ForecastFeatureBuilder,
    ModelBundle,
    ModelRegistry,
    TemporalServeState,
    TemporalSpec,
    assert_bundle_quality,
    assert_no_leakage,
    baseline_prediction,
    drift_report,
    load_bundle,
    make_supervised,
    mltrack,
    save_bundle,
    serve_forecast,
    validate_temporal_frame,
)
from notebook_data import SEED, SOURCE, artifact_dir, daily_series_data
from notebook_report import compile_pdf, markdown_table, save_figure

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X has feature names")
warnings.filterwarnings("ignore", message=".*force_all_finite.*")
pd.set_option("display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

REPORT_DIR = artifact_dir("forecasting-report")
print(f"SKSurrogate {SKSurrogate.__version__} from {Path(SKSurrogate.__file__).parent}")
print(f"scikit-learn {sklearn.__version__} | pandas {pd.__version__} | numpy {np.__version__}")
print(f"seed={SEED}  data source={SOURCE!r}  artifacts -> {REPORT_DIR.resolve()}")

## 1. The data

Provenance first, then the shape of the series. Two of these numbers shape everything that
follows: the target has gaps, and the exogenous promotion flag is only known *at the
origin* — which is what makes it legal to use as a feature.

In [ ]:
dataset = daily_series_data()
frame = dataset.frame.copy()
TARGET = dataset.target

span = frame["date"].max() - frame["date"].min()
provenance = pd.DataFrame(
    [
        ("source", dataset.source),
        ("description", dataset.description),
        ("rows", len(frame)),
        ("time span", f"{frame['date'].min().date()} .. {frame['date'].max().date()} ({span.days + 1} days)"),
        ("sampling", "daily, complete date axis"),
        ("target", TARGET),
        ("target mean / sd", f"{frame[TARGET].mean():.2f} / {frame[TARGET].std(ddof=1):.2f}"),
        ("target gaps", f"{int(frame[TARGET].isna().sum())} rows ({100 * frame[TARGET].isna().mean():.2f} %)"),
        ("promo gaps", f"{int(frame['promo'].isna().sum())} rows"),
        ("promo rate", f"{frame['promo'].fillna(0).mean():.3f} (share of days with a promotion)"),
    ],
    columns=["field", "value"],
)
display(provenance)
for key, value in dataset.notes.items():
    print(f"{key}: {value}")
display(frame.head(3))

In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(13, 8.6), constrained_layout=True)

axes[0].plot(frame["date"], frame[TARGET], linewidth=0.8, color="C0")
gaps = frame.loc[frame[TARGET].isna(), "date"]
axes[0].scatter(gaps, np.full(len(gaps), float(np.nanmin(frame[TARGET]))), marker="x",
                color="C3", s=26, label=f"{len(gaps)} missing days")
axes[0].set(ylabel="demand", title=f"Daily demand, {frame['date'].min():%Y-%m} to {frame['date'].max():%Y-%m}")
axes[0].legend(fontsize=8, loc="upper left")

weekly = frame.assign(day=frame["date"].dt.day_name()).groupby("day")[TARGET].mean()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
axes[1].bar(order, weekly.reindex(order).to_numpy(), color="C2")
axes[1].set(ylabel="mean demand", title="Weekly profile: the seasonality a lag-7 feature can see")

monthly = frame.assign(month=frame["date"].dt.month).groupby("month")[TARGET]
axes[2].boxplot([group.dropna().to_numpy() for _, group in monthly],
                tick_labels=[f"{m:02d}" for m, _ in monthly])
axes[2].set(xlabel="month", ylabel="demand", title="Monthly distribution: level changes across the year")

figure.suptitle("Exploratory view of a daily series", fontsize=13)
save_figure(figure, REPORT_DIR, "eda-overview")
display(figure)

### Autocorrelation, computed here rather than imported

A correlogram is a few lines of linear algebra, and writing them out is the difference
between *reading* a plot and *trusting* one. Two estimators, both standard:

$$\hat\rho_k = \frac{\sum_{t=1}^{n-k}(x_t-\bar x)(x_{t+k}-\bar x)}{\sum_{t=1}^{n}(x_t-\bar x)^2},
\qquad \phi_{kk} \;\text{by Durbin–Levinson recursion on}\; \hat\rho_1,\dots,\hat\rho_K .$$

The dashed band is the white-noise reference $\pm 1.96/\sqrt{n}$: a bar inside it is what
sampling noise alone produces.

The estimators below are checked against ``statsmodels`` where it is installed: the
autocorrelation agrees to $10^{-16}$ and the partial autocorrelation to $5\times10^{-14}$
with ``pacf(..., method="ywm")``, which is what this notebook used to plot.

In [ ]:
# Sample autocorrelation at lags 1..nlags: the biased estimator, denominator n.
def acf(values, nlags):
    x = np.asarray(values, dtype=float)
    x = x[~np.isnan(x)]
    demeaned = x - x.mean()
    denominator = float(demeaned @ demeaned)
    return np.array(
        [float(demeaned[k:] @ demeaned[: len(demeaned) - k]) / denominator for k in range(1, nlags + 1)]
    )


# Partial autocorrelation by Durbin-Levinson recursion on the sample ACF.
def pacf(values, nlags):
    rho = np.concatenate([[1.0], acf(values, nlags)])
    phi = np.zeros((nlags + 1, nlags + 1))
    out = np.zeros(nlags)
    for order in range(1, nlags + 1):
        numerator = rho[order] - sum(phi[order - 1, j] * rho[order - j] for j in range(1, order))
        denominator = 1.0 - sum(phi[order - 1, j] * rho[j] for j in range(1, order))
        phi[order, order] = numerator / denominator
        for j in range(1, order):
            phi[order, j] = phi[order - 1, j] - phi[order, order] * phi[order - 1, order - j]
        out[order - 1] = phi[order, order]
    return out


# Stems plus the +/-1.96/sqrt(nobs) white-noise band.
def draw_correlogram(stem, ax, title, nobs):
    lags = np.arange(1, len(stem) + 1)
    ax.vlines(lags, 0, stem, color="C0", linewidth=1.4)
    ax.plot(lags, stem, "o", color="C0", markersize=2.6)
    ax.axhline(0, color="k", linewidth=0.8)
    for sign in (1, -1):
        ax.axhline(sign * 1.96 / np.sqrt(nobs), color="C3", linestyle="--", linewidth=0.9)
    ax.set_title(title)
    ax.set_xlabel("lag (days)")

In [ ]:
# ACF/PACF need a dense series, so this is computed on the interpolated copy and labelled
# as such: the diagnostic describes the seasonal structure, it is not an evaluation.
dense = frame.set_index("date")[TARGET].interpolate(limit=3)
dense_values = dense.dropna()
nobs = len(dense_values)

figure, axes = plt.subplots(2, 1, figsize=(11, 6.4), constrained_layout=True)
draw_correlogram(acf(dense_values, 60), axes[0], "Autocorrelation (interpolated copy, 60 lags)", nobs)
draw_correlogram(pacf(dense_values, 40), axes[1], "Partial autocorrelation (Levinson-Durbin)", nobs)
figure.suptitle("Weekly spikes at 7, 14, 21 and an annual shoulder", fontsize=13)
save_figure(figure, REPORT_DIR, "acf-pacf")
display(figure)

weekly_acf = dense.dropna().autocorr(lag=7)
annual_acf = dense.dropna().autocorr(lag=364)
print(f"ACF(lag 7) = {weekly_acf:.3f} | ACF(lag 364) = {annual_acf:.3f}")

In [ ]:
# Does the promotion move demand? A difference of means, not a causal effect: promotions
# were scheduled by the business, not randomised, and they cluster in high-demand periods.
promo = frame.dropna(subset=[TARGET]).assign(promo_flag=frame["promo"].fillna(0).astype(int))
promo_summary = promo.groupby("promo_flag")[TARGET].agg(["mean", "std", "size"]).rename(
    index={0: "no promotion", 1: "promotion"}
)
display(promo_summary.round(2))
print("difference of means:", round(promo_summary.loc["promotion", "mean"] - promo_summary.loc["no promotion", "mean"], 3))
print("share of promoted days:", round(float(promo["promo_flag"].mean()), 3))

## 2. The temporal contract

`TemporalSpec` is the declaration that turns a table into a time series: which column is
time, what frequency is expected, and how far ahead the model must forecast. Everything
downstream — the feature builder, the splitter guard, the backtest, the serving state —
reads this declaration instead of guessing.

Validating it is cheap and catches the expensive mistakes (unsorted rows, duplicate
timestamps, an off-grid axis, a frequency that does not match the data).

In [ ]:
HORIZON = 7          # one week ahead: an operational planning cycle
SEASONALITY = 7      # weekly seasonality, used by MASE and by the seasonal-naive baseline

spec = TemporalSpec(
    time_column="date",
    series_id=None,      # a single series; a panel would name its id column here
    freq="D",            # declared, so gaps in the *axis* are detected, not smoothed over
    horizon=HORIZON,
    order="increasing",
    gap_tolerance=0,
)
report = validate_temporal_frame(frame, spec)
print(json.dumps({key: report[key] for key in
                  ("ok", "time_column", "horizon", "declared_freq", "rows", "time_kind", "time_source")},
                 indent=2))
print("warnings:", report["warnings"])
print("errors  :", report["errors"])

### Gaps: a policy, stated and measured

The date axis is complete but 17 demand values and 37 promotion flags are missing. There
are two defensible policies and one indefensible one:

* **drop** every row a feature needs — correct, and it costs history: 470 of 1095 rows
  cannot be used with a 28-day rolling window;
* **repair short gaps** — interpolate runs of one to three days, leave longer holes alone,
  and say so;
* ~~forward-fill the whole column~~ — it fabricates a flat stretch and every lag feature
  then reads it as data.

The notebook repairs short gaps and reports the row cost of both policies below.

In [ ]:
def gap_runs(series):
    # Lengths of consecutive missing runs, so "17 gaps" can be read as "how long".
    missing = series.isna().to_numpy()
    lengths, run = [], 0
    for flag in missing:
        if flag:
            run += 1
        elif run:
            lengths.append(run)
            run = 0
    if run:
        lengths.append(run)
    return pd.Series(lengths).value_counts().sort_index()


gap_table = pd.concat(
    [gap_runs(frame[TARGET]).rename("target count"), gap_runs(frame["promo"]).rename("promo count")],
    axis=1,
).fillna(0).astype(int)
gap_table.index.name = "run length (days)"
gap_table["runs"] = gap_table.sum(axis=1)
display(gap_table)
target_runs = gap_runs(frame[TARGET])
print("longest target gap:", int(target_runs.index.max()), "day(s) | promo runs by length:",
      {int(length): int(count) for length, count in gap_runs(frame["promo"]).items()})

In [ ]:
FEATURES = dict(
    lags=[7, 14, 21, 56],                 # every lag is >= the horizon, by contract
    seasonal_lags=[28],                   # four weeks back: same weekday, same phase
    rolling=[{"window": 7, "stats": ["mean"]}, {"window": 28, "stats": ["mean", "std"]}],
    calendar=["dow", "month", "doy", "is_weekend"],
    fourier={"periods": [7, 365.25], "terms": 2},
    exogenous=["promo"],                  # known at the origin, so it may be a feature
)

raw_design = make_supervised(frame, target=TARGET, spec=spec, **FEATURES)

repaired = frame.sort_values("date").reset_index(drop=True)
repaired[TARGET] = repaired[TARGET].interpolate(limit=3)
repaired["promo"] = repaired["promo"].fillna(0.0)
repaired_design = make_supervised(repaired, target=TARGET, spec=spec, **FEATURES)

policy_table = pd.DataFrame(
    [
        {"policy": "drop incomplete rows", "design_rows": len(raw_design.X),
         "rows_lost": raw_design.dropped_rows, "target_gaps_left": int(frame[TARGET].isna().sum())},
        {"policy": "interpolate gaps of <= 3 days", "design_rows": len(repaired_design.X),
         "rows_lost": repaired_design.dropped_rows, "target_gaps_left": int(repaired[TARGET].isna().sum())},
    ]
)
display(policy_table)
print("design columns:", len(repaired_design.X.columns), "| features built:", len(repaired_design.feature_specs))

Repairing one- and two-day holes more than doubles the usable history (625 → 1032 rows) and
the remaining 63 lost rows are the warm-up period the 56-day lag needs. That is the whole
argument: it is not about a nicer number, it is about the warm-up cost being paid once
instead of on every origin.

## 3. Leakage-safe features

The builder derives every feature from the past *within the series*, shifted by at least the
horizon. Two things are worth verifying rather than trusting:

1. **the naming**, which states the shift (`demand__lag_7`, `demand__roll_mean_28`);
2. **the property**, tested by perturbation: corrupt everything after a cut and confirm the
   features of rows at or before that cut do not move (`assert_no_leakage`).

The third check is the guard itself — a lag shorter than the horizon is an error, not a
warning.

In [ ]:
builder = ForecastFeatureBuilder(spec=spec, target=TARGET, **FEATURES)
builder.fit(repaired)
design = builder.build(repaired)

feature_table = pd.DataFrame(
    [
        {"feature": item.name, "op": item.op, "source": item.column,
         "reads_from": f"t-{item.shift}" if item.shift else "origin time",
         "window": item.window}
        for item in design.feature_specs
    ]
)
display(feature_table.head(14))
print(f"{len(feature_table)} features from {len(repaired)} rows -> design {design.X.shape}")

leakage = assert_no_leakage(builder.transform, repaired, spec, cuts=3)
print("perturbation check:", json.dumps(leakage, default=str)[:400])

In [ ]:
# The guard: a feature that reads inside the forecast window is refused outright.
try:
    ForecastFeatureBuilder(spec=spec, target=TARGET, lags=[1, 3]).fit(repaired)
except ValueError as exc:
    print("lag 1 refused:", str(exc)[:180])

try:
    ForecastFeatureBuilder(
        spec=spec, target=TARGET, lags=[7],
        rolling=[{"window": 7, "stats": ["mean"], "center": True}],
    ).fit(repaired)
except (TypeError, ValueError) as exc:
    print("centered window refused:", str(exc)[:180])

## 4. The baseline discipline

MASE divides the forecast error by the in-sample error of seasonal naive. So **MASE = 1 is
seasonal naive**, 0.5 is twice as good, and 1.2 means the model should not be deployed.

Every backtest below scores the baselines on *the same origins* as the model — same folds,
same horizon, same MASE scale — because a baseline computed on a different window is not a
comparison. This is a property of the return value, not of the analyst's diligence.

In [ ]:
# A worked example of the two baselines, so MASE=1 stops being abstract.
history = repaired[TARGET].to_numpy()[:120]
naive_forecast = baseline_prediction(history, "naive", HORIZON, SEASONALITY)
seasonal_forecast = baseline_prediction(history, "seasonal_naive", HORIZON, SEASONALITY)
five_day = baseline_prediction(history, "seasonal_naive", 5, SEASONALITY)

print(f"last observed value            : {history[-1]:.2f}")
print(f"naive, 7 steps ahead           : {naive_forecast:.2f}")
print(f"seasonal naive, 7 steps ahead  : {seasonal_forecast:.2f}")
print(f"seasonal naive, 5 steps ahead  : {five_day:.2f}  (reads t-3: 7 - ((5-1) mod 7) = 3)")
print("with horizon == seasonality, seasonal naive collapses to naive:",
      f"steps_back = m - ((h-1) mod m) = {SEASONALITY} - (({HORIZON}-1) % {SEASONALITY}) = 1")

## 5. Registering the task and refusing a shuffled split

The tracker stores the declared spec, and from then on it enforces order-respecting
splitters. The refused call below is the single most valuable line in this notebook: it is
the difference between a walk-forward estimate and a fantasy.

In [ ]:
workdir = artifact_dir(".notebook-work", root=REPORT_DIR)
tracker = mltrack("notebook-forecasting", db_name=str(workdir / "mltrace.db"))
tracker.RegisterData(repaired, TARGET, partition="train")
stored_spec, partition_reports = tracker.SetTemporalSpec(spec, partitions=["train"])
print("spec stored:", json.dumps(stored_spec)[:160])
print("partition validation:", {name: entry["ok"] for name, entry in partition_reports.items()})

try:
    tracker.SetCV(KFold(n_splits=4, shuffle=True, random_state=SEED))
except ValueError as exc:
    print("shuffled KFold refused:", str(exc)[:220])

print("accepted:", tracker.SetCV(TimeSeriesSplit(n_splits=4)))

## 6. Model selection with the backtest as the metric

Three candidates, three inductive biases: a regularised linear model on Fourier and
calendar terms, an extremely randomised forest, and gradient boosting. Each is evaluated by
the same rolling-origin backtest, so the comparison is between *procedures*, not fits.

The result is a good reminder that "gradient boosting wins" is not a law: this series is
additive and seasonal, and a tree cannot extrapolate a level.

In [ ]:
CANDIDATES = {
    "ridge (alpha=1.0)": Ridge(alpha=1.0),
    "extra trees (200)": ExtraTreesRegressor(n_estimators=200, random_state=SEED, n_jobs=1),
    "lightgbm (300 x 0.05)": None,   # filled in below if the optional dependency exists
}
try:
    from lightgbm import LGBMRegressor

    CANDIDATES["lightgbm (300 x 0.05)"] = LGBMRegressor(
        n_estimators=300, learning_rate=0.05, verbose=-1, random_state=SEED, n_jobs=1
    )
except Exception as exc:  # pragma: no cover - optional dependency
    print("lightgbm unavailable, comparing two candidates only:", exc)

import time

candidate_rows, candidate_results = [], {}
for label, estimator in CANDIDATES.items():
    started = time.perf_counter()
    result = tracker.backtest(
        estimator,
        horizon=HORIZON,
        n_splits=10,
        step=HORIZON,
        seasonality=SEASONALITY,
        feature_kwargs=FEATURES,
        baselines=("naive", "seasonal_naive"),
        store=False,
    )
    candidate_results[label] = result
    candidate_rows.append(
        {
            "candidate": label,
            "mase": result["pooled"]["mase"],
            "mae": result["pooled"]["mae"],
            "smape": result["pooled"]["smape"],
            "evaluated_rows": result["pooled"]["rows"],
            "fit_seconds": round(time.perf_counter() - started, 2),
            "beats_seasonal_naive": result["ranking"]["winner_beats_baseline"],
        }
    )

candidate_table = pd.DataFrame(candidate_rows).sort_values("mase").reset_index(drop=True)
baseline_row = {
    "candidate": "seasonal naive (baseline)",
    "mase": candidate_results[candidate_table.loc[0, "candidate"]]["baselines"]["seasonal_naive"]["mase"],
    "mae": candidate_results[candidate_table.loc[0, "candidate"]]["baselines"]["seasonal_naive"]["mae"],
    "smape": candidate_results[candidate_table.loc[0, "candidate"]]["baselines"]["seasonal_naive"]["smape"],
    "evaluated_rows": candidate_results[candidate_table.loc[0, "candidate"]]["baselines"]["seasonal_naive"]["rows"],
    "fit_seconds": np.nan,
    "beats_seasonal_naive": None,
}
display(pd.concat([candidate_table, pd.DataFrame([baseline_row])], ignore_index=True).round(3))

figure, ax = plt.subplots(figsize=(8.6, 3.6), constrained_layout=True)
bars = ax.barh(candidate_table["candidate"], candidate_table["mase"], color="C0")
ax.axvline(baseline_row["mase"], color="C3", linestyle="--", linewidth=1.2,
           label=f"seasonal naive = {baseline_row['mase']:.2f}")
ax.invert_yaxis()
ax.set(xlabel="MASE over the same origins (lower is better)", title="Backtest comparison")
for bar, value in zip(bars, candidate_table["mase"]):
    ax.text(value + 0.02, bar.get_y() + bar.get_height() / 2, f"{value:.3f}", va="center", fontsize=9)
ax.legend(fontsize=8, loc="lower right")
save_figure(figure, REPORT_DIR, "candidate-comparison")
display(figure)
print("selected:", candidate_table.loc[0, "candidate"])

## 7. The final backtest, and what the intervals are calibrated on

One backtest with prediction intervals, on the selected model:

* `alpha=0.1` targets 90 % coverage;
* `interval_method="asymmetric"` uses the empirical 5 % and 95 % residual quantiles, which
  matters when errors are skewed;
* the calibration is fitted on the *held-out residuals of the walk-forward*, so it inherits
  the same no-peeking property as the point forecast.

`ranking` then answers the only question a metric can answer on its own: does the winner
beat the baseline, and by how much.

In [ ]:
best_label = candidate_table.loc[0, "candidate"]
best_estimator = CANDIDATES[best_label]

final = tracker.backtest(
    best_estimator,
    horizon=HORIZON,
    n_splits=10,
    step=HORIZON,
    seasonality=SEASONALITY,
    feature_kwargs=FEATURES,
    baselines=BASELINE_METHODS,
    alpha=0.10,
    interval_method="asymmetric",
)
print("stored under:", final["artifact_key"])
print("folds:", len(final["folds"]), "| fits:", final["fits"], "| evaluated rows:", final["pooled"]["rows"])

metric_rows = [
    {"forecast": "model", **{key: final["pooled"][key] for key in ("mase", "mae", "rmse", "smape", "bias")}},
]
for method, metrics in final["baselines"].items():
    if metrics.get("rows"):
        metric_rows.append(
            {"forecast": method, **{key: metrics[key] for key in ("mase", "mae", "rmse", "smape", "bias")}}
        )
metric_table = pd.DataFrame(metric_rows)
display(metric_table.round(3))

ranking = pd.DataFrame(final["ranking"]["ranked"])
display(ranking.round(3))
print("winner beats the baseline:", final["ranking"]["winner_beats_baseline"],
      "| baseline used:", final["ranking"]["best"]["baseline"])

In [ ]:
diagnostics = final["intervals"]["diagnostics"]
calibration = final["intervals"]["calibration"]
print("calibration:", json.dumps({key: calibration[key] for key in
                                  ("method", "alpha", "target_step", "n_residuals")}, indent=2))
print("calibrated offsets (lower, upper):",
      round(calibration["lower_offset"], 3), round(calibration["upper_offset"], 3))
print(json.dumps({key: diagnostics[key] for key in
                  ("sample_size", "empirical_coverage", "nominal_coverage", "mean_interval_width",
                   "mean_interval_score")}, indent=2))
print("coverage Wilson interval:", diagnostics["coverage_interval"])

origins = pd.DataFrame(final["origins"])
origins["target_time"] = pd.to_datetime(origins["target_time"])
origins["origin_time"] = pd.to_datetime(origins["origin_time"])
origins["error"] = origins["y_pred"] - origins["y_true"]
print("origins table:", origins.shape)

In [ ]:
last_fold = origins[origins["fold"] == origins["fold"].max()].sort_values("target_time")
history_window = repaired[repaired["date"] <= last_fold["origin_time"].iloc[0]].tail(90)

figure, axes = plt.subplots(2, 1, figsize=(12.6, 7.4), constrained_layout=True,
                            gridspec_kw={"height_ratios": [1.4, 1]})
axes[0].plot(history_window["date"], history_window[TARGET], color="C0", linewidth=1, label="observed")
axes[0].plot(last_fold["target_time"], last_fold["y_true"], "o-", color="C2", label="actual (held out)")
axes[0].plot(last_fold["target_time"], last_fold["y_pred"], "s--", color="C3", label="forecast, 7 days ahead")
axes[0].set(ylabel="demand", title="Last walk-forward fold: one 7-day forecast from one origin")
axes[0].legend(fontsize=8, loc="upper left")

axes[1].axhline(0, color="k", linewidth=0.8)
for fold, group in origins.groupby("fold"):
    axes[1].plot(group.sort_values("target_time")["target_time"], group.sort_values("target_time")["error"],
                 marker="o", markersize=3, linewidth=0.8, alpha=0.75, label=f"fold {fold}")
axes[1].set(xlabel="target time", ylabel="error (prediction - actual)", title="Every origin, every step")
axes[1].legend(fontsize=6, ncol=5, loc="upper left")
figure.suptitle("Walk-forward evidence", fontsize=13)
save_figure(figure, REPORT_DIR, "backtest-fan")
display(figure)

## 8. Accuracy and interval width as a function of the horizon

A single 7-day backtest answers one question. The honest follow-up is how both quantities
behave as the horizon grows — with a *fixed* evaluation length, otherwise a longer horizon
simply gets more rows and the comparison is meaningless.

Note what the table does and does not say: each row is a separately fitted direct model for
that step, so this is "how hard is step *h*", not "how does one model degrade".

In [ ]:
EVALUATION_DAYS = 70
sweep_rows = []
for horizon in range(1, HORIZON + 1):
    n_splits = max(int(round(EVALUATION_DAYS / horizon)), 3)
    result = tracker.backtest(
        best_estimator,
        horizon=horizon,
        n_splits=n_splits,
        step=horizon,
        seasonality=SEASONALITY,
        feature_kwargs=FEATURES,
        baselines=("naive", "seasonal_naive"),
        alpha=0.10,
        interval_method="asymmetric",
        # Every origin is a fit under refit="every_fold", so a 1-day horizon over a
        # 70-day evaluation needs 70 fits. The default ceiling is 64 by design: it
        # makes a runaway grid fail loudly instead of running for an hour.
        cost_limit=200,
        store=False,
    )
    stats = result["intervals"]["diagnostics"]
    sweep_rows.append(
        {
            "horizon_days": horizon,
            "origins": n_splits,
            "evaluated_rows": result["pooled"]["rows"],
            "mase": result["pooled"]["mase"],
            "mae": result["pooled"]["mae"],
            "seasonal_naive_mase": result["baselines"]["seasonal_naive"]["mase"],
            "coverage_90": stats["empirical_coverage"],
            "mean_interval_width": stats["mean_interval_width"],
        }
    )
sweep_table = pd.DataFrame(sweep_rows)
display(sweep_table.round(3))

figure, axes = plt.subplots(1, 3, figsize=(14.4, 3.9), constrained_layout=True)
axes[0].plot(sweep_table["horizon_days"], sweep_table["mase"], "o-", label="model")
axes[0].plot(sweep_table["horizon_days"], sweep_table["seasonal_naive_mase"], "s--", label="seasonal naive")
axes[0].set(xlabel="horizon (days)", ylabel="MASE", title="Accuracy vs horizon")
axes[0].legend(fontsize=8)

axes[1].plot(sweep_table["horizon_days"], sweep_table["mean_interval_width"], "o-", color="C3")
axes[1].set(xlabel="horizon (days)", ylabel="mean 90% interval width", title="Uncertainty vs horizon")

axes[2].plot(sweep_table["horizon_days"], sweep_table["coverage_90"], "o-", color="C2")
axes[2].axhline(0.90, color="k", linestyle="--", linewidth=1, label="nominal 0.90")
axes[2].set(xlabel="horizon (days)", ylabel="empirical coverage", title="Coverage vs horizon")
axes[2].set_ylim(0.0, 1.02)
axes[2].legend(fontsize=8)
figure.suptitle("Direct models per horizon, equal evaluation length", fontsize=13)
save_figure(figure, REPORT_DIR, "horizon-sweep")
display(figure)

print(f"MASE across horizons: {sweep_table['mase'].min():.3f} .. {sweep_table['mase'].max():.3f}"
      f" (seasonal naive {sweep_table['seasonal_naive_mase'].min():.3f} .. {sweep_table['seasonal_naive_mase'].max():.3f})")
print(f"coverage across horizons: {sweep_table['coverage_90'].min():.3f} .. {sweep_table['coverage_90'].max():.3f}"
      f" (nominal 0.90) | mean width {sweep_table['mean_interval_width'].min():.2f} .. {sweep_table['mean_interval_width'].max():.2f}")
print("Each row is a separately fitted direct model for that step, so the model's own error")
print("does not compound along a path; what changes with the horizon is how much of the")
print("future is genuinely unknown. Flatness here is a result, not a bug.")

## 9. Residual diagnostics: is anything left in the errors?

Two questions a forecast report should answer, and rarely does:

* **Is the error autocorrelated?** If residuals correlate with their own lags, the model has
  left structure on the table (usually a missed seasonal term or a level change).
* **Is the error biased by day, by level, or by promotion?** A model that is fine on average
  and wrong every Saturday is not fine.

In [ ]:
residual_series = origins.sort_values("target_time").set_index("target_time")["error"]
figure, axes = plt.subplots(1, 3, figsize=(14.4, 3.9), constrained_layout=True)

residual_values = residual_series.dropna()
draw_correlogram(
    acf(residual_values, 28), axes[0], "Residual autocorrelation (28 lags)", len(residual_values)
)

by_weekday = (
    origins.assign(day=origins["target_time"].dt.day_name())
    .groupby("day")["error"]
    .mean()
    .reindex(order)
)
axes[1].bar(order, by_weekday.to_numpy(), color="C1")
axes[1].axhline(0, color="k", linewidth=0.8)
axes[1].set(ylabel="mean error", title="Bias by weekday")
axes[1].tick_params(axis="x", rotation=45, labelsize=8)

axes[2].scatter(origins["y_true"], origins["error"], s=14, alpha=0.7)
axes[2].axhline(0, color="k", linewidth=0.8)
axes[2].set(xlabel="actual demand", ylabel="error", title="Error vs level")
figure.suptitle("Where the forecast is wrong", fontsize=13)
save_figure(figure, REPORT_DIR, "residual-diagnostics")
display(figure)

lag1 = residual_series.autocorr(lag=1)
lag7 = residual_series.autocorr(lag=7)
band = 1.96 / np.sqrt(len(residual_series))
print(f"residual ACF(1) = {lag1:.3f}, ACF(7) = {lag7:.3f}, 95% white-noise band = +/-{band:.3f}")
print("weekly structure left in the residuals:", abs(lag7) > band)
print(f"worst absolute error: {origins['error'].abs().max():.2f} on {origins.loc[origins['error'].abs().idxmax(), 'target_time']:%Y-%m-%d}")

## 10. Monitoring without labels

Demand forecasting never has labels at the moment the forecast is made, so monitoring has to
compare *distributions*. `drift_report` in window mode compares the last N rows of a
reference period against the last N of the current one, per column, and reports PSI plus
range and missingness checks.

The demo shifts the level by 15 % over a late-autumn window — the kind of regime change a
promotion calendar or a new customer segment produces.

In [ ]:
reference_window = repaired.tail(180).head(90).reset_index(drop=True)
shifted = repaired.tail(90).reset_index(drop=True).copy()
shifted[TARGET] = shifted[TARGET] * 1.15

monitoring = drift_report(
    reference_window,
    shifted,
    time_column="date",
    reference_window=90,
    current_window=90,
    model_version="notebook-forecasting",
)
drift_table = pd.DataFrame(
    [
        {
            "column": column,
            "psi": entry["value"],
            "status": "alert" if entry["value"] is not None and entry["value"] >= monitoring["thresholds"]["psi"] else "ok",
        }
        for column, entry in monitoring["drift"]["numeric"].items()
    ]
)
display(drift_table.round(3))
print("alerts:", json.dumps(monitoring["alerts"], indent=1))

figure, axes = plt.subplots(1, 2, figsize=(12.6, 3.8), constrained_layout=True)
axes[0].hist(reference_window[TARGET], bins=25, alpha=0.6, label="reference (90 days)", color="C0")
axes[0].hist(shifted[TARGET], bins=25, alpha=0.6, label="current (level +15%)", color="C3")
axes[0].set(xlabel="demand", ylabel="days", title="Demand distribution, reference vs current")
axes[0].legend(fontsize=8)

axes[1].bar(drift_table["column"], drift_table["psi"], color=["C3" if s == "alert" else "C0" for s in drift_table["status"]])
axes[1].axhline(monitoring["thresholds"]["psi"], color="k", linestyle="--", linewidth=1,
                label=f"PSI threshold {monitoring['thresholds']['psi']}")
axes[1].set(ylabel="PSI", title="Per-column population stability index")
axes[1].legend(fontsize=8)
figure.suptitle("What monitoring can say before any label arrives", fontsize=13)
save_figure(figure, REPORT_DIR, "monitoring")
display(figure)

## 11. Package it, gate it, serve it

A forecasting bundle is not a tabular bundle with dates. It carries the serving geometry:

* the `ForecastFeatureBuilder`, so raw history can be turned into the same features at
  inference time;
* the `TemporalServeState`, so `serve_forecast` knows the spec, the target and the strategy;
* the `IntervalCalibration` from the backtest, so the served forecast has intervals with a
  stated provenance.

The bundle is gated on MASE, so a model that loses to seasonal naive cannot be promoted.

In [ ]:
# The pipeline is fitted on the design the builder produced, so the served model is the
# same estimator on the same features -- not an unfitted clone.
fitted_model = clone(best_estimator).fit(design.X, design.y)
pipeline = Pipeline([("features", builder), ("model", fitted_model)])
state = TemporalServeState.from_parts(spec, TARGET, builder, strategy="direct")

baseline_mase = final["baselines"]["seasonal_naive"]["mase"]
# The quality gate takes a *minimum* per metric, which cannot express "MASE is better when
# smaller". The gate therefore uses a skill score: 1 - MASE / seasonal-naive MASE, so 0 is
# parity with the baseline and 0.25 means a quarter less error.
skill_vs_baseline = float(1.0 - final["pooled"]["mase"] / baseline_mase)
print(f"skill vs seasonal naive: {skill_vs_baseline:.3f}")

bundle = ModelBundle(
    pipeline,
    task_name="notebook-forecasting",
    schema={column: {"dtype": str(repaired[column].dtype)} for column in repaired.columns},
    metrics={
        **{key: final["pooled"][key] for key in ("mase", "mae", "rmse", "smape") if final["pooled"][key] is not None},
        "skill_vs_seasonal_naive": skill_vs_baseline,
    },
    interval_calibration=calibration,
    temporal=state.to_dict(),
    metadata={
        "baseline_scores": final["baselines"],
        "backtest": {
            "artifact_key": final["artifact_key"],
            "config": final["config"],
            "ranking": final["ranking"],
        },
        "features": FEATURES,
        "evaluation_protocol": "rolling-origin backtest, 10 origins, step = horizon",
    },
    owner="notebook",
    run_id="notebook-forecasting-run",
    dependencies={},
)
bundle.record_audit_event(
    "training",
    estimator=best_label,
    metrics=bundle.metrics,
    baseline_scores={"seasonal_naive": final["baselines"]["seasonal_naive"]["mase"]},
)
bundle_path = save_bundle(bundle, workdir / "forecast.bundle")
loaded = load_bundle(bundle_path)
print("bundle:", bundle_path)
print("temporal block:", json.dumps({key: loaded.temporal[key] for key in ("format", "strategy", "target")}))
print("metrics recorded:", {key: round(value, 3) for key, value in loaded.metrics.items()})

# Gate on the skill score (must beat seasonal naive), not on MASE itself.
quality = assert_bundle_quality(bundle_path, metric_thresholds={"skill_vs_seasonal_naive": 0.0})
print("quality gates:", [(check["name"], check.get("metric"), check.get("passed")) for check in quality["checks"]])

registry = ModelRegistry(workdir / "registry")
model_version = registry.register(loaded)
DeploymentApprovalGate(registry).promote(
    loaded.task_name, model_version, "production",
    approvers=["forecast-owner"], quality_report=quality,
)
print("promoted ->", model_version)

### The serving contract, and why it is strict

`serve_forecast` refuses a history that cannot support the declared features. With gaps in
the last two months, a 28-day rolling window has nothing to average — and the error says
exactly which column and which origin, instead of returning a confident number built from
NaNs.

That is the right failure. The repaired history below is the same policy as section 2,
applied to the serving window.

In [ ]:
last_observation = pd.Timestamp(repaired["date"].max())
# Build the future axis with date_range rather than date arithmetic: numpy datetime64 plus
# a timedelta goes through a deprecated generic unit path in this stack.
CUTOFF = pd.date_range(end=last_observation, periods=HORIZON + 1, freq="D")[0]
future_window = pd.DataFrame(
    {"date": pd.date_range(start=last_observation, periods=HORIZON + 1, freq="D")[1:],
     "promo": 0.0}
)

try:
    serve_forecast(loaded, frame[frame["date"] <= CUTOFF], horizon=HORIZON, future_exogenous=future_window)
except ValueError as exc:
    print("raw (gappy) history refused:", str(exc)[:300])

serving_history = repaired[repaired["date"] <= CUTOFF]
forecast = serve_forecast(loaded, serving_history, horizon=HORIZON, future_exogenous=future_window)
forecast_table = forecast.to_frame()
display(forecast_table.round(2))
print("interval note:", (forecast.interval_note or "")[:220])

In [ ]:
figure, ax = plt.subplots(figsize=(12.6, 4.4), constrained_layout=True)
tail = serving_history.tail(120)
ax.plot(tail["date"], tail[TARGET], color="C0", linewidth=1, label="observed history")
ax.fill_between(forecast_table["time"], forecast_table["lower"], forecast_table["upper"],
                color="C3", alpha=0.2, label="90% interval (calibrated on backtest residuals)")
ax.plot(forecast_table["time"], forecast_table["mean"], "s--", color="C3", label="forecast")
ax.axvline(forecast_table["time"].iloc[0], color="k", linestyle=":", linewidth=0.9)
ax.set(xlabel="date", ylabel="demand", title="Seven-day-ahead forecast from the last origin")
ax.legend(fontsize=8, loc="upper left")
save_figure(figure, REPORT_DIR, "serving")
display(figure)

# Roll forward: append one week of actuals and re-forecast from the new origin.
roll_history = repaired
roll_future = pd.DataFrame(
    {"date": pd.date_range(start=last_observation, periods=HORIZON + 1, freq="D")[1:],
     "promo": 0.0}
)
rolled = serve_forecast(loaded, roll_history, horizon=HORIZON, future_exogenous=roll_future).to_frame()
print("rolled origin:", forecast_table["time"].max().date(), "->", rolled["time"].min().date())
display(rolled.round(2))

## 12. The report

In [ ]:
generated_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def fmt(value, digits=3):
    return "n/a" if value is None or (isinstance(value, float) and not np.isfinite(value)) else f"{value:.{digits}f}"


report_sections = [
    ("Summary", f"""
Daily demand, {len(repaired)} observations, forecast {HORIZON} days ahead with direct models
and a 10-origin walk-forward backtest.

| estimate | value | baseline | scope |
| --- | --- | --- | --- |
| MASE | {fmt(final['pooled']['mase'])} | {fmt(final['baselines']['seasonal_naive']['mase'])} (seasonal naive) | 70 held-out days |
| MAE | {fmt(final['pooled']['mae'], 2)} | {fmt(final['baselines']['seasonal_naive']['mae'], 2)} | demand units |
| sMAPE % | {fmt(final['pooled']['smape'], 2)} | {fmt(final['baselines']['seasonal_naive']['smape'], 2)} | percentage |
| 90% interval coverage | {fmt(diagnostics['empirical_coverage'])} | nominal {fmt(diagnostics['nominal_coverage'], 2)} | {diagnostics['sample_size']} rows |

MASE is scaled by the in-sample seasonal-naive error, so 1.0 *is* seasonal naive. The
selected model scores {fmt(final['pooled']['mase'])}, i.e. {skill_vs_baseline * 100:.0f} % less
error than seasonal naive on the same origins.
"""),
    ("Data", f"""
{dataset.source}: {dataset.description}. {len(frame)} daily rows,
{frame['date'].min():%Y-%m-%d} to {frame['date'].max():%Y-%m-%d}. Target `{TARGET}`;
exogenous `promo`, known at the origin.

Missing values: {int(frame[TARGET].isna().sum())} demand rows, {int(frame['promo'].isna().sum())}
promotion flags. Policy: interpolate runs of at most three days, treat a missing promotion
flag as no promotion. Cost of the alternative, measured:

{markdown_table(policy_table, digits=0)}
"""),
    ("Temporal contract", f"""
time_column `date`, frequency `{spec.freq}`, horizon {HORIZON}, single series.
Validation: {json.dumps({key: report[key] for key in ('ok', 'time_kind', 'rows')})}.
A shuffled KFold is refused by the tracker because it ignores observation order.
"""),
    ("Model selection", f"""
Every candidate was scored by the same 10-origin walk-forward backtest on the same MASE
scale, with the baselines scored on the same origins.

{markdown_table(candidate_table, digits=3)}
"""),
    ("Evaluation", f"""
{markdown_table(metric_table, digits=3)}

Ranking against the baseline:

{markdown_table(ranking, digits=3)}
"""),
    ("Horizon sweep", f"""
Direct models fitted per step with an equal evaluation length ({EVALUATION_DAYS} days).

{markdown_table(sweep_table, digits=3)}
"""),
    ("Diagnostics", f"""
Residual autocorrelation: ACF(1) = {fmt(lag1, 3)}, ACF(7) = {fmt(lag7, 3)}, 95 % white-noise
band = +/-{fmt(band, 3)}. Largest absolute error: {fmt(origins['error'].abs().max(), 2)}.
"""),
    ("Monitoring", f"""
Window-mode drift over the last 90 days against a reference window, with the level shifted
by 15 %.

{markdown_table(drift_table, digits=3)}
"""),
    ("Serving", f"""
The bundle carries the feature builder, the serving geometry and the interval calibration.
A history that cannot support the declared features is refused with the column and origin
named; a repaired history produces:

{markdown_table(forecast_table, digits=2)}
"""),
    ("Limitations", f"""
Synthetic data with an additive seasonal structure, which is why a linear model wins here;
a series with a strong trend or a level shift would favour a different bias. Intervals are
calibrated on backtest residuals for one target step and do not model error compounding
along the multi-step path, so early steps may be covered more often than nominal. The
promotion effect is a difference of means on observational data, not a causal estimate. One
series, one frequency, one horizon; a panel or an hourly series would add structure this
notebook does not exercise.
"""),
]

report_markdown = "# Forecasting report: daily demand\n\n" + "\n\n".join(
    f"## {title}\n\n{body.strip()}" for title, body in report_sections
)
report_markdown += "\n\nGenerated at " + generated_at + " from a seeded, reproducible notebook run.\n"

tables = {
    "candidates": candidate_table,
    "metrics": metric_table,
    "ranking": ranking,
    "horizon-sweep": sweep_table,
    "drift": drift_table,
    "forecast": forecast_table,
    "policy": policy_table,
    "features": feature_table,
}
for name, table in tables.items():
    table.to_csv(REPORT_DIR / f"{name}.csv", index=False)

(REPORT_DIR / "analysis-metadata.json").write_text(
    json.dumps(
        {
            "generated_at": generated_at,
            "dataset": {"source": dataset.source, "description": dataset.description, "notes": dataset.notes},
            "seed": SEED,
            "temporal_spec": spec.to_dict(),
            "features": {key: value for key, value in FEATURES.items()},
            "rows": {"raw": len(frame), "modelled": len(repaired), "design": len(design.X)},
            "gap_policy": policy_table.to_dict(orient="records"),
            "metrics": metric_table.to_dict(orient="records"),
            "baselines": final["baselines"],
            "interval_calibration": calibration,
            "interval_diagnostics": diagnostics,
            "ranking": final["ranking"],
            "backtest_artifact_key": final["artifact_key"],
            "model_version": model_version,
            "limitations": [
                "Synthetic additive seasonal series; a linear model wins by construction.",
                "Intervals calibrated per target step; no error compounding modelled.",
                "Promotion effect is associational, not causal.",
            ],
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

report_figures = {}
for stem in (
    "eda-overview",
    "acf-pacf",
    "candidate-comparison",
    "backtest-fan",
    "horizon-sweep",
    "residual-diagnostics",
    "monitoring",
    "serving",
):
    pdf_path = REPORT_DIR / f"{stem}.pdf"
    if pdf_path.exists():
        report_figures[f"figures/{stem}.pdf"] = pdf_path.read_bytes()

from api.reports import figure_latex

inserts = {
    # figure_latex returns a list of LaTeX lines; two figures under one heading are
    # concatenated, because a second dict entry with the same key would win silently.
    ("Data", None): (
        figure_latex("figures/eda-overview.pdf", "Level, weekly profile and monthly distribution.")
        + figure_latex("figures/acf-pacf.pdf", "Autocorrelation and partial autocorrelation (interpolated copy).")
    ),
    ("Model selection", None): figure_latex("figures/candidate-comparison.pdf", "MASE per candidate against the seasonal-naive baseline."),
    ("Evaluation", None): figure_latex("figures/backtest-fan.pdf", "Last fold forecast and the error of every origin."),
    ("Horizon sweep", None): figure_latex("figures/horizon-sweep.pdf", "MASE, interval width and coverage against the horizon."),
    ("Diagnostics", None): figure_latex("figures/residual-diagnostics.pdf", "Residual autocorrelation, weekday bias and error against level."),
    ("Monitoring", None): figure_latex("figures/monitoring.pdf", "Reference vs current demand distribution and per-column PSI."),
    ("Serving", None): figure_latex("figures/serving.pdf", "Served forecast with calibrated 90% intervals."),
}

report_path = REPORT_DIR / "report.md"
report_path.write_text(report_markdown, encoding="utf-8")
pdf, note = compile_pdf(
    report_markdown,
    task="notebook-forecasting",
    title="Daily demand: end-to-end forecasting",
    subtitle=f"SKSurrogate {SKSurrogate.__version__} / seed {SEED}",
    figures=report_figures,
    inserts=inserts,
    generated_at=generated_at,
)
if pdf is not None:
    (REPORT_DIR / "report.pdf").write_bytes(pdf)
print("PDF:", note)
print("artifacts:", sorted(path.name for path in REPORT_DIR.iterdir() if path.is_file()))

In [ ]:
from IPython.display import Markdown

Markdown(report_markdown)

## What this notebook does *not* establish

* **The linear model winning is a property of this series.** Additive seasonality plus
  Fourier terms is almost exactly the generator's structure; a trending or regime-switching
  series would reverse the ranking.
* **MASE compares against seasonal naive on the same origins**, not against the best possible
  forecast. Beating it is necessary, not sufficient.
* **Intervals are marginal and per step.** Coverage is calibrated on backtest residuals for
  the design horizon, and the note attached to every served forecast says so; multi-step
  error compounding is not modelled.
* **The promotion effect is associational.** Promotions were not randomised, and the
  difference of means mixes the promotion with the demand regime it was scheduled in.
* **One series, one frequency.** The same machinery handles panels via `series_id` and other
  frequencies via `freq`, but neither is exercised here.

That closes the trio: [classification](01_classification_end_to_end.ipynb) and
[regression](02_regression_end_to_end.ipynb) for the tabular contracts, this one for the
temporal contract.

In [ ]:
tracker.close()
print("done.")